# Detector Calibration from Lab Data

Single-photon detectors are the heart of every QKD receiver. **SNSPDs**
(Superconducting Nanowire Single-Photon Detectors) offer > 85% detection efficiency
and ultra-low dark counts at cryogenic temperatures, while **InGaAs SPADs** operate
near room temperature but suffer from afterpulsing. Accurate calibration of detection
efficiency vs. bias current and dark-count rate vs. temperature is essential for
realistic QKD simulations.

In [ ]:
from photonstrust.calibrate.detector_calibration import (
    fit_snspd_efficiency_curve,
    fit_dcr_temperature,
)

In [ ]:
import numpy as np

# Generate synthetic SNSPD calibration data
bias_currents = np.linspace(5, 20, 15)
# Sigmoid model: PDE = max_pde / (1 + exp(-k*(I - I0)))
true_pde = 0.85 / (1 + np.exp(-0.8 * (bias_currents - 12)))
noise = np.random.default_rng(42).normal(0, 0.01, len(bias_currents))
measured_pde = np.clip(true_pde + noise, 0, 1)

cal = fit_snspd_efficiency_curve(bias_currents.tolist(), measured_pde.tolist())
print(f"Max PDE: {cal.max_pde:.3f}")
print(f"Optimal bias: {cal.optimal_bias_uA:.1f} \u00b5A")

## Temperature-Dependent Dark Count Rates

Dark counts follow an **Arrhenius-like** relationship with temperature: lowering the
detector temperature exponentially suppresses thermal dark counts. The function
`fit_dcr_temperature` fits measured DCR vs. temperature data to extract the activation
energy and baseline dark count rate.

In [ ]:
# Synthetic DCR vs temperature data (Arrhenius model)
temperatures_K = np.linspace(1.5, 4.0, 12)
# DCR ~ A * exp(-Ea / (kB * T))
true_dcr = 10.0 * np.exp(-5.0 / temperatures_K)
dcr_noise = np.random.default_rng(99).normal(0, 0.5, len(temperatures_K))
measured_dcr = np.clip(true_dcr + dcr_noise, 0.01, None)

dcr_fit = fit_dcr_temperature(temperatures_K.tolist(), measured_dcr.tolist())
print(f"Fitted DCR model: {dcr_fit}")

## Try It Yourself

- Increase the noise standard deviation to 0.05 and see how the fitted parameters
  change — this simulates a noisier lab measurement.
- Try fitting InGaAs afterpulsing data using the afterpulsing calibration utilities
  in `photonstrust.calibrate`.
- Feed the calibration results back into `simulate_qkd_link` via a custom detector
  configuration to see how calibration accuracy affects key-rate predictions.